# 第 21 节: RL 调试方法论

## 学习目标
1. 掌握 RL 训练中的常见故障模式
2. 学会系统性地诊断问题
3. 能根据训练曲线判断可能的原因
4. 完成一个故意写错的 PPO 调试练习

## 1. RL 调试的特殊挑战

RL 调试比监督学习更难，因为:
- 非平稳性: 数据分布随策略改变而变化
- 延迟反馈: 奖励可能在很多步后才出现
- 多组件交互: Actor x Critic x Buffer x Env
- 随机性大: 同一配置不同种子结果可能完全不同
- 没有正确答案: 没有标准测试集

## 2. 调试检查清单

### 第一步: 环境检查
- Observation 的 shape 和 range 是否正确?
- Reward scale 是否合理? (推荐 [-1, 1] 或 [0, 1])
- Terminated vs Truncated 是否正确处理?
- 随机策略的 episode return 基线是多少?

### 第二步: 网络检查
- 网络输入/输出维度是否匹配?
- 初始化是否合理? (推荐正交初始化)
- 梯度是否正常流动? (grad_norm)

### 第三步: 算法检查
- Advantage 是否 detach?
- Old log prob 是否正确保存?
- Return 和 Advantage 是否混淆?
- Bootstrap 边界条件是否正确?

### 第四步: 训练过程
- Reward 是否上升?
- Loss 是否在合理范围?
- Entropy 是否下降太快?
- 有没有 NaN?

## 3. 常见故障模式及诊断

### 故障 1: Reward 不上升
可能原因: 学习率太小/太大、网络太浅、训练不够久
诊断: 先跑随机策略看 return 范围，再逐步调大 lr

### 故障 2: Loss 震荡剧烈
可能原因: batch size 太小、lr 太大、return 未标准化
诊断: 增大 batch size、降低 lr、添加 return normalization

### 故障 3: Entropy 过快下降
可能原因: 策略过早收敛到确定性策略
诊断: 增大 entropy coefficient、减小 lr

### 故障 4: NaN 出现
可能原因: 梯度爆炸、log(0)、除零
诊断: 添加 grad clipping、检查 log_prob 计算、添加 eps

### 故障 5: Critic loss 很大且不下降
可能原因: return scale 太大、网络不够强
诊断: reward normalization、更深的 Critic

### 故障 6: Episode return 上升但 evaluation return 不上升
可能原因: 训练时用了 exploration，eval 时没开
诊断: 确保 eval 用 deterministic 模式

## 4. 诊断练习: 找出下面 PPO 代码的 8 个错误

In [ ]:
# 这是一个有 8 个错误的 PPO 实现
# 试试在不看答案的情况下找出所有错误!

import torch, numpy as np

class BuggyPPO:
    def __init__(self):
        self.gamma = 0.99
        self.clip_epsilon = 0.2

    def update(self, states, actions, old_log_probs, advantages, returns):
        logits, values = self.network(states)

        # BUG 1: 没有 detach advantages!
        new_log_probs = torch.log_softmax(logits, dim=-1)
        action_log_probs = new_log_probs.gather(1, actions.unsqueeze(-1)).squeeze(-1)

        ratio = torch.exp(action_log_probs - old_log_probs)

        # BUG 2: ratio 没有 clamp, 应该 min(surr1, surr2)
        policy_loss = -(ratio * advantages).mean()

        # BUG 3: 用 returns 而非 values 做 value loss
        # BUG 4: 没有 normalize advantages
        value_loss = (values.squeeze(-1) - returns).pow(2).mean()

        # BUG 5: 没有 entropy bonus
        # BUG 6: total_loss 没有加 value_loss
        total_loss = policy_loss

        # BUG 7: 没有 gradient clipping
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()

        # BUG 8: 只用了一轮 update (应该多轮 minibatch)
        return {}

print("试试找出上面的 8 个错误!")

<details><summary><b>点击查看答案 (8 个错误)</b></summary>

1. Advantages 未 detach: Actor loss 中 advantages 应该 .detach()
2. ratio 未 clip: 应该用 min(ratio*A, clip(ratio, 1-e, 1+e)*A)
3. Value loss 用了 returns 而非 values: 应该是 (values - returns)^2
4. Advantages 未标准化: 应该 advantages = (adv - mean) / (std + 1e-8)
5. 缺少 Entropy bonus: 应加 -c2 * entropy 到 total loss
6. Total loss 缺少 value loss: 应加 c1 * value_loss
7. 缺少 Gradient clipping: 应 clip_grad_norm_(params, max_norm)
8. 单轮 update: PPO 应该多轮 minibatch SGD
</details>

## 5. 正确的调试策略

1. 从小处开始: 先用最简单的环境 (CartPole) 验证
2. 一次改一个: 每次只改一个超参数
3. 看多个指标: 不只是 reward, 还有 loss, entropy, KL, clip fraction
4. 多跑几个种子: 至少 3-5 个随机种子
5. 对比已知正确实现: 用成熟库做 baseline
6. 可视化一切: 不要只看数字

## 6. 练习
1. 修复上面的 BuggyPPO
2. 在你的 PPO 实现中故意引入 error, 看是否能从训练曲线中识别
3. 收集至少 5 个不同种子的训练曲线, 计算均值和标准差

---
下一节: 22_experiment_design.ipynb